<a href="https://colab.research.google.com/github/AkhileshSR/AkhileshSR/blob/main/feb2026_langgraph_crash_course.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LangGraph Crash Course

<img src="https://drive.google.com/uc?id=1bwlbmjbPZLXBizw5Ekl7icGgjiDM7RQP" alt="Alt text" width="700"/>


LangGraph is a framework designed for building **complex, stateful LLM agent and multi-agent applications**.

Unlike traditional linear workflows, LangGraph uses a **graph-based architecture** that gives developers fine-grained control over agent behavior—enabling features like conditional logic, tool prioritization, looping, and shared memory.

It's ideal for creating **robust, production-ready AI systems** that require precision and adaptability.

The first thing we need to do, is to install a bunch of **Python libraries** we'll need throughout this crash course.

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langchain_openai langchain_core langchain_community langgraph

# Understanding LangGraph Structure

Let's start by understanding the fundamental components of LangGraph and how to create a simple graph.


<img src="https://drive.google.com/uc?id=1HlCxqbbJ_xyIFCSimyPKAS_MVcGnne7J" alt="Alt text" width="700"/>


Remember that LangGraph is based on three fundamental components: **state**, **nodes**, and **edges**. Let's start with the state.

## 1. State

Start by defining the State of the graph.

This state schema acts as the input structure for all Nodes and Edges within the graph.

In [ ]:
from typing_extensions import TypedDict

class State(TypedDict):
    graph_state: str
    message: str

## 2. Nodes

**Nodes are simply Python functions.**

Each node takes the state as its first positional argument, based on the previously defined `TypedDict` schema.

Since the state includes a `graph_state` key, each node can access it using `state['graph_state']`.

> Each node returns an updated value for graph_state, and by default, this new value will overwrite the existing one in the state.

In [ ]:
def node_1(state):
    print("---Node 1---")
    return {"graph_state": state['graph_state'] +" Welcome", "message": state['message'] + " Namaskaram"}

def node_2(state):
    print("---Node 2---")
    return {"graph_state": state['graph_state'] +" to the Future Proof India!", "message": state['message'] + " Vanakkam"}

def node_3(state):
    print("---Node 3---")
    return {"graph_state": state['graph_state'] +" to India!", "message": state['message'] + " Ola"}

def node_4(state):
    print("---Node 4---")
    return {"graph_state": state['graph_state'] +" to Hyderabad!", "message": state['message'] + " Hello"}

## 3. Edges

Edges define the connections between nodes in the graph.

* **Normal edges** are used when you always want to transition from one node to another—for example, from `node_1` to `node_2`.

* **Conditional edges** allow for **dynamic routing** based on logic. These are implemented as functions that evaluate the current state and return the name of the next node to execute.

In [ ]:
import random
from typing import Literal


def decide_node(state) -> Literal["node_2", "node_3", "node_4"]:

    user_input = state['graph_state']

    rand = random.random()

    if rand < 0.5:
        return "node_2"
    elif rand < 0.8:
        return "node_3"
    else:
        return "node_4"

## 4. Graph Construction

Now it's time to build the graph using the components we've defined.

We'll use the `StateGraph` class to create the graph structure.

First, initialize a `StateGraph` with the `State` schema we defined earlier.

> Then, add your nodes and connect them with edges.

Use the special `START` node to define the entry point of the graph—this is where user input enters the system.

The `END` node marks the terminal point where the graph finishes execution.

Once all nodes and edges are added, compile the graph to validate its structure.

You can also visualize the resulting graph as a **Mermaid diagram** for a clearer view of the workflow.

In [ ]:
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END

# Build graph
builder = StateGraph(State)

# Defining the nodes
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("node_3", node_3)
builder.add_node("node_4", node_4)

# Logic - Defining the edges
builder.add_edge(START, "node_1")
builder.add_conditional_edges("node_1", decide_node)
builder.add_edge("node_2", END)
builder.add_edge("node_3", END)
builder.add_edge("node_4", END)

# Compile the Graph
graph = builder.compile()

# View
display(Image(graph.get_graph().draw_mermaid_png()))

## 5. Graph Execution

The compiled graph conforms to the `Runnable` protocol, which defines a standard interface for executing LangChain components.

One of the key methods in this interface is `.invoke()`.

You start by passing an input dictionary, such as `{"graph_state": "Hi there, it's Sridhar!"}`, to set the initial value of the graph state.

When `.invoke()` is called, execution begins at the `START` node.

The graph then proceeds through the defined nodes (`node_1`, `node_2`, `node_3`), following the structure you built.

A conditional edge determines whether the flow goes from `node_1` to `node_2` or `node_3`, based on a 50/50 logic split.

Each node receives the current state, processes it, and returns an updated value, which replaces the previous graph_state.

Execution continues along the graph until the END node is reached, signaling completion.

In [ ]:
graph.invoke({"graph_state" : "Hi there, it's Sridhar!", "message": "Namaste"})

Congratulations, now we have an idea of how LangGraph works. Time to see what kind of LLM applications we can build with this framework!

# Example 2

In [ ]:
# Minimal LangGraph practice: Support Ticket Triage + Reply (with 1 retry loop)
# LLM-agnostic (no model calls). Pure State + Nodes + Edges + Compile + Invoke.

from typing import TypedDict, Optional
from langgraph.graph import StateGraph, END


# 1) STATE
class State(TypedDict, total=False):
    msg: str
    category: Optional[str]       # billing | bug | feature | other
    action: Optional[str]         # answer | clarify | escalate
    reply: Optional[str]
    ok: Optional[bool]
    attempts: int


# 2) NODES (pure state transforms)

def classify(s: State) -> State:
    m = s["msg"].lower()
    if any(k in m for k in ["refund", "invoice", "payment", "charged"]):
        cat = "billing"
    elif any(k in m for k in ["error", "bug", "crash", "not working"]):
        cat = "bug"
    elif any(k in m for k in ["feature", "request", "add"]):
        cat = "feature"
    else:
        cat = "other"
    return {"category": cat}


def decide(s: State) -> State:
    m = s["msg"].lower()
    cat = s["category"]

    if cat == "bug" and any(k in m for k in ["down", "data loss", "security"]):
        action = "escalate"
    elif cat in ["billing", "bug"] and ("order" not in m and "id" not in m):
        action = "clarify"
    else:
        action = "answer"

    return {"action": action}


def draft(s: State) -> State:
    a, c = s["action"], s["category"]

    if a == "escalate":
        r = "Thanks for reporting this. We’re escalating to our engineering team now and will update you shortly."
    elif a == "clarify":
        r = "Thanks! Please share your Order ID and a short description of what happened, so I can help quickly."
    else:
        if c == "billing":
            r = "Thanks! I can help with billing. Please share your Order ID and what you were charged for."
        elif c == "bug":
            r = "Thanks for reporting the issue. Please share steps to reproduce and your device/browser details."
        elif c == "feature":
            r = "Thanks for the feature request! I’ve noted it and will share it with the product team."
        else:
            r = "Thanks for reaching out! Please share a bit more detail so I can help you faster."

    return {"reply": r}


def check(s: State) -> State:
    attempts = s.get("attempts", 0) + 1
    reply = (s.get("reply") or "").strip()

    # Simple quality rule: decent length + contains a next-step cue
    ok = (len(reply) >= 25) and any(k in reply.lower() for k in ["please", "share", "will", "update"])

    return {"ok": ok, "attempts": attempts}


def improve(s: State) -> State:
    # One simple improvement pass
    return {"reply": (s["reply"] + " If possible, share a screenshot too.")}


# 3) ROUTERS (control flow)

def route_after_check(s: State) -> str:
    if s.get("ok"):
        return "done"
    if s.get("attempts", 0) >= 2:
        return "done"
    return "improve"


# 4) GRAPH WIRING

g = StateGraph(State)

g.add_node("classify", classify)
g.add_node("decide", decide)
g.add_node("draft", draft)
g.add_node("check", check)
g.add_node("improve", improve)

g.set_entry_point("classify")

g.add_edge("classify", "decide")
g.add_edge("decide", "draft")
g.add_edge("draft", "check")

g.add_conditional_edges(
    "check",
    route_after_check,
    {"improve": "improve", "done": END},
)

g.add_edge("improve", "check")

app = g.compile()

In [ ]:
display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
# 5) INVOKE (example)

if __name__ == "__main__":
    state: State = {"msg": "Hi, Can you clarify how to use the zoom transcript.", "attempts": 0}
    out = app.invoke(state)

    print("Category:", out.get("category"))
    print("Action:", out.get("action"))
    print("Attempts:", out.get("attempts"))
    print("\nReply:\n", out.get("reply"))

# Example 3

In [ ]:
# -----------------------------
# 1) Define State
# -----------------------------
class EmailState(TypedDict):
    topic: str
    draft: str
    word_count: int


# -----------------------------
# 2) Define Nodes (pure transforms)
# -----------------------------
def draft_email(state: EmailState) -> EmailState:
    """
    Draft a simple email (stubbed, LLM-agnostic).
    Produces: draft, word_count
    """
    topic = state["topic"]
    draft = (
        "Hello,\n\n"
        f"I’m writing regarding {topic}. "
        "Could you please share your thoughts and the next steps? "
        "I appreciate your time.\n\n"
        "Regards,\n"
        "Sridhar"
    )
    return {**state, "draft": draft, "word_count": len(draft.split())}


def shorten_email(state: EmailState) -> EmailState:
    """
    Shorten the email (stubbed).
    Produces: draft, word_count
    """
    topic = state["topic"]
    shortened = (
        "Hello,\n\n"
        f"Quick note on {topic}. "
        "Can you confirm next steps?\n\n"
        "Regards,\n"
        "Sridhar"
    )
    return {**state, "draft": shortened, "word_count": len(shortened.split())}


# -----------------------------
# 3) Define Edges (control flow)
# -----------------------------
def check_length(state: EmailState) -> str:
    """
    Router: decide what happens next based on state.
    """
    # tweak this threshold to see the loop behavior clearly
    if state["word_count"] > 25:
        return "shorten"
    return "end"


# -----------------------------
# 4) Assemble the Graph
# -----------------------------
def build_graph():
    builder = StateGraph(EmailState)

    # Nodes
    builder.add_node("draft", draft_email)
    builder.add_node("shorten", shorten_email)

    # Entry
    builder.set_entry_point("draft")

    # Conditional edge from "draft"
    builder.add_conditional_edges(
        "draft",
        check_length,
        {
            "shorten": "shorten",
            "end": END,
        },
    )

    # Loop back: after shortening, draft again (re-check length)
    builder.add_edge("shorten", "draft")

    # -----------------------------
    # 5) Compile
    # -----------------------------
    return builder.compile()

In [ ]:
# -----------------------------
# 6) Invoke
# -----------------------------
def main():
    graph = build_graph()

    initial_state: EmailState = {
        "topic": "project deadline extension",
        "draft": "",
        "word_count": 0,
    }

    result = graph.invoke(initial_state)

    print("\n=== FINAL EMAIL DRAFT ===\n")
    print(result["draft"])
    print("\nWords:", result["word_count"])
    display(Image(graph.get_graph().draw_mermaid_png()))


if __name__ == "__main__":
    main()

# Building LLM Applications with LangGraph

Our **Telegram Agent** will follow a pattern known as the **Router**, in which the LLM decides which workflow to follow.

**This Router is an evolution of static LLM Chains.**

Let's take a look at how to implement each one in the following sections.

> Don't forget about the Levels of Autonomy in LLM Applications! 👇


<img src="https://drive.google.com/uc?id=1eQtFcYG3Elfdl0_jCGEgM7bUTiv-o213" alt="Alt text" width="700"/>
